In [21]:
import os
import re
import shutil
import tempfile
import asyncio
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent
from google.adk.runners import Runner
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool, ToolContext
from google.adk.sessions import InMemorySessionService
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.apps.app import App, ResumabilityConfig
from google.genai import types
from dotenv import load_dotenv

async def initialize_adk_model():
    """Initializes the Google ADK LLM agent with Gemini model."""
    load_dotenv()  # Load environment variables from .env file
    api_key = os.getenv("GOOGLE_API_KEY")
    github_token = os.getenv("GITHUB_TOKEN")
    os.environ["GOOGLE_API_KEY"] = api_key  # This keeps the compatibility with how ADK might expect the API key internally
    os.environ["GITHUB_TOKEN"] = github_token
    if not api_key:
        raise ValueError("GEMINI_API_KEY environment variable not set.")
    if not github_token:
        raise ValueError("GITHUB_TOKEN environment variable not set.")

In [2]:
def read_file(file_path: str) -> str:
    """
    Reads the content of a file.

    Args:
        file_path: The path to the file.

    Returns:
        The content of the file as a string.
    """
    if not os.path.exists(file_path):
        return f"Error: File not found at {file_path}"
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file: {e}"


def read_directory_files(directory_path: str) -> dict[str, str]:
    """
    Reads the content of all files in a directory and its subdirectories.

    Args:
        directory_path: The path to the directory.

    Returns:
        A dictionary where keys are file paths and values are file contents.
    """
    if not os.path.isdir(directory_path):
        return {"error": f"Error: Directory not found at {directory_path}"}
    
    file_contents = {}
    for root, _, files in os.walk(directory_path):
        for file in files:
            file_path = os.path.join(root, file)
            # Skip .git directory files
            if '.git' in file_path.split(os.sep):
                continue
            file_contents[file_path] = read_file(file_path)
            
    return file_contents


async def read_github_repository(repo_url: str) -> dict:
    """
    Clones a public GitHub repository and reads the content of all its files.
    Returns the temporary directory path where the repository was cloned, along with file contents.

    Args:
        repo_url: The full URL of the GitHub repository to clone.

    Returns:
        A dictionary with keys 'file_contents' (dictionary of file paths and contents) and 'temp_dir' (path to the temporary directory),
        or an error message.
    """
    print(f"Debug: read_github_repository received URL: {repo_url}") # Debug print
    # Validate the GitHub URL - more flexible regex
    if not re.match(r"https://github\.com/([^/]+)/([^/]+)", repo_url):
        return {"error": "Invalid GitHub repository URL provided."}

    temp_dir = tempfile.mkdtemp()
    try:
        # Construct the git clone command
        command = f"git clone --depth 1 {repo_url} ."
        
        # Execute the command in the temporary directory
        process = await asyncio.create_subprocess_shell(
            command,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
            cwd=temp_dir
        )
        
        stdout, stderr = await process.communicate()

        if process.returncode != 0:
            # Clean up on clone failure
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
            return {"error": f"Failed to clone repository. Error: {stderr.decode()}"}

        # Read the files from the cloned repository
        file_contents = read_directory_files(temp_dir)
        return {"file_contents": file_contents, "temp_dir": temp_dir}

    except Exception as e:
        # Clean up on unexpected error
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
        return {"error": f"An unexpected error occurred: {e}"}
            

def cleanup_temp_directory(directory_path: str) -> dict:
    """
    Removes a temporary directory and its contents.

    Args:
        directory_path: The path to the temporary directory to remove.

    Returns:
        A dictionary indicating success or an error message.
    """
    if not os.path.isdir(directory_path):
        return {"error": f"Error: Directory not found at {directory_path}"}
    try:
        shutil.rmtree(directory_path)
        return {"success": f"Successfully removed directory: {directory_path}"}
    except Exception as e:
        return {"error": f"Error removing directory {directory_path}: {e}"}


async def get_linting_score(path: str) -> dict:
    """
    Runs pylint on a given Python file or directory and returns its linting score.

    Args:
        path: The path to the Python file or directory to lint.

    Returns:
        A dictionary containing the linting score as a float and any errors,
        or an error message if the path is not found or pylint fails.
    """
    if not os.path.exists(path):
        return {"error": f"Error: Path not found at {path}"}
    
    # Check if it's a file and ensure it's a Python file
    if os.path.isfile(path) and not path.endswith('.py'):
        return {"error": "Error: Not a Python file."}

    try:
        # Run pylint as a subprocess with the absolute path
        absolute_path = os.path.abspath(path)
        # If it's a directory, pylint will try to lint modules inside it.
        # Adding --recursive=y is safer for modern pylint, but standard pylint <dir> often works too.
        command = f"pylint {absolute_path}"
        
        process = await asyncio.create_subprocess_shell(
            command,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE
        )
        stdout, stderr = await process.communicate()

        pylint_output = stdout.decode()
        pylint_error = stderr.decode()

        # Pylint often returns non-zero exit codes for linting issues, so we don't strictly fail on returncode != 0
        if process.returncode != 0 and "No such file or directory" in pylint_error:
             return {"error": "Pylint is not installed or not found. Please install it using 'pip install pylint'."}
        
        # Extract the score using a regular expression
        match = re.search(r"Your code has been rated at ([-+]?\d*\.\d+|\d+)/10", pylint_output)
        if match:
            score_out_of_10 = float(match.group(1))
            percentage_score = score_out_of_10 * 10  # Convert to percentage
            return {"linting_score": percentage_score}
        else:
            return {
                "linting_score": 0.0, 
                "message": "Pylint ran but no score was found (possibly no python files or fatal errors).", 
                "pylint_output": pylint_output[:500] # Return partial output for debugging
            }

    except Exception as e:
        return {"error": f"An unexpected error occurred during linting: {e}"}

In [3]:
correctness_assessor = LlmAgent(
    model="gemini-2.0-flash",
    name="correctness_assessor",
    description="An agent specialized in code correctness assessment.",
    instruction="""
        You are a highly experienced coding expert specializing in code correctness assessment. Your task is to meticulously evaluate the provided code or codebase based on the following criteria:

        1.  **Functionality:** Assess if the code robustly delivers its requirements, effectively handles errors, manages invalid inputs gracefully, and considers common and edge cases.
        2.  **Security:** Identify any potential security vulnerabilities, common exploits, or insecure practices.
        3.  **Resource Efficiency & Memory Optimization:** Evaluate the code's efficiency in terms of CPU usage, memory consumption, and overall resource management.
        4.  **Test Coverage:** If a repository is provided, determine the presence and quality of automated tests (unit, integration, etc.). Note if tests are absent or poorly implemented.

        For each of these four points, provide a concise analysis (2-3 sentences max) and a corresponding percentage score (0-100%). Finally, synthesize these into an overall code correctness score (0-100%). 
        Present your assessment in a clear, structured Markdown format, with a dedicated section for each point and the final overall score.
        """,
    tools=[ FunctionTool(read_file),
            FunctionTool(read_directory_files),
            FunctionTool(read_github_repository),
            FunctionTool(get_linting_score),
            FunctionTool(cleanup_temp_directory),
            ],
)


style_assessor = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="style_assessor",
    description="An agent specialized in code style assessment.",
    instruction="""You are an expert code style assessor. Your task is to provide a structured code style assessment report for the provided code or repository.

            **For any code input (file or directory):**
            Evaluate the code based on:
                -   **Readability:** Descriptive naming, consistent formatting, comments.
                -   **Maintainability & Organization:** Structure, modularity, large functions, duplication.
                -   **Python Linting:** Use the score obtained from `get_linting_score`.
                -   **Repository Best Practices:** Presence of `README.md`, `.gitignore`, `requirements.txt`, etc.

            Provide a concise analysis (2-3 sentences) and a percentage score (0-100%) for each aspect. Conclude with an overall code style score (0-100%). Present your assessment in clear Markdown with dedicated sections 
            for each point and the final overall score. **Do not output raw file contents.**
            """,
    tools=[ FunctionTool(read_file),
            FunctionTool(read_directory_files),
            FunctionTool(read_github_repository),
            FunctionTool(get_linting_score),
            FunctionTool(cleanup_temp_directory),
            ],
)


description_generator = LlmAgent(
    model="gemini-2.5-flash",
    name="description_generator",
    description="An agent specialized in generating analyses and description for the code.",
    instruction="""You are an expert software engineer with a deep understanding of artificial intelligence, specializing in 
                    generating comprehensive and clear code descriptions. Your task is to analyze the provided code (file, directory, or repository) 
                    and generate a detailed, easy-to-understand description.

                    Your description should:
                    1.  **Overview:** Start with a high-level summary of the code's purpose and overall architecture, mentioning the languages and libraries used, an any configuration or scripts modules.
                    2.  **Component Breakdown:** Identify key files, classes, and functions, explaining their individual roles and how they contribute to the overall system.
                    3.  **Functionality Summary:** Conclude with a concise, single-line summary of the code's primary functionality.

                    Ensure your response is structured in Markdown for readability. Do not include raw code content in your output.
                    """,
    tools=[ FunctionTool(read_file),
            FunctionTool(read_directory_files),
            FunctionTool(read_github_repository),
            FunctionTool(get_linting_score),
            FunctionTool(cleanup_temp_directory),
            ],
)




In [4]:
parallel_parsing_agent = ParallelAgent(
    name="ParallelCodeAssessmentAgent",
    sub_agents=[description_generator, correctness_assessor, style_assessor],
    description="""receive the code, generate assessments and textual description, then communicate it to the 
    report generator and improvement recommender""",
)

In [5]:
improvement_recommender = LlmAgent(
    model="gemini-3.0-pro",
    name="improvement_recommender",
    description="recommends valid improvements based on the report received",
    instruction="""recommend valid improvements for the code described in the report received and return 
    them in a list.""",
)

In [6]:
code_broker_backend = SequentialAgent(
    name="CodeBroker_backend",
    sub_agents=[parallel_parsing_agent , improvement_recommender],
    description="generate assessment, textual description and recommended improvement.",
)

In [ ]:
# Example: Defining the basic identity
report_generator = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="report_generator",
    description="Answers user questions about the capital city of a given country.",
    instruction="""You are an agent that provides the capital city of a country... (previous instruction text)""",

)

In [ ]:
def pretty_interface():
    return

In [22]:
async def main():
        code_file_path = "<path>"
        code_dir_path = "<path>"
        repo_http_path = "https://github.com/Samir-atra/Emotion_estimation_from_video_footage_with_LSTM_ML_algorithm"
        USER_ID = "explorer"
        session_id = "explorer_session"

        code_broker_app = App(
            name="code_broker",
            root_agent=code_broker_backend,
            resumability_config=ResumabilityConfig(is_resumable=True),
        )

        agent = await initialize_adk_model()
        session_service = InMemorySessionService()

        # Create runner with the resumable app
        coding_runner = Runner(
        app=code_broker_app,  # Pass the app instead of the agent
        session_service=session_service,
        )
        
        async for event in coding_runner.run_async(
            user_id=USER_ID, 
            session_id=session_id, 
            new_message=types.Content(parts=[types.Part(text=f"provide a report for the repository at:{repo_http_path}")])
        ):
            print(f"Event: {event.type}")  # Print event type for debugging
            
            # Check if this is a final response event
            if hasattr(event, 'content') and event.content:
                final_response = event.content
                print(f"Response: {event.content}")

        print("\nFinal Assessment Report:")
        print(final_response if final_response else "No response received")

if __name__ == "__main__":
    await main()

/tmp/ipykernel_273999/3854651742.py:11: UserWarning: [EXPERIMENTAL] ResumabilityConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  resumability_config=ResumabilityConfig(is_resumable=True),


ValueError: Session not found: explorer_session